<h1>Chapter 5 - Tools</h1>
<i>Giving an Agent access to the Environment through Tool Usage</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 5 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [1]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [2]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM - `Gemma 3`

At the beginning of every chapter, we start by choosing the LLM that we want to use. In this notebook, we will explore how to enable tool calling for models that do not have this capability. It will show you how to use JSON and the difficulties when trying to parse it. As such, the model that we will be using throughout this chapter is Gemma 3, a model that does not have native tool calling capabilities.

In [5]:
from openai import OpenAI
from illustrated_agents.chapters.ch2 import LLM

# Ollama through OpenAI API
client = OpenAI(base_url="http://localhost:11434/v1/", api_key="no_key")
llm = LLM(model="gemma3:12b", client=client)

# Llama.cpp server
# client = OpenAI(base_url="http://localhost:8080/v1/", api_key="no_key")
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", client=client)

# LM Studio
# client = OpenAI(base_url="http://localhost:1234/v1/", api_key="no_key")
# llm = LLM(model="gemma-3-12B-it", client=client)

# Google's Gemini / Gemma
# client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="YOUR_GEMINI_API_KEY")
# llm = LLM(model="gemini-2.5-flash", client=client)

## 2 - Adding **`Tools`**

In the previous chapter, we added the `Memory` module to your `TinyAgent`. In this chapter, we will cover how to give it access to tools:

![../images/ch5.png](../images/ch5.png)

Tool usage, as covered in the book, has a wide range of methodologies to choose from (function calling, JSON, MCP, skills, etc.). Implementing them can therefore be a bit tricky, especially when you have are using an LLM that was not trained specifically for one of those techniques. Fortunately, LLMs are quite capable these days and even smaller models can follow instructions quite accurately. This gives us a bit of freedom with prompting techniques.

Throughout this notebook we are going with the following type of workflow:

* Tool Creation
* Tool Definition
* Tool Selection
* Tool Calling
* Tool Output Processing

![../images/ch5_tools.png](../images/ch5_tools.png)



## 3 - Tool Creation

Before we create a tool registry for your `TinyAgent`, let's first explore how we can create a tool ourselves. In its most basic form, a tool is nothing more than a function which may take in some arguments and outputs a string. In the following code block, we create a basic calculator that only adds value and a `get_weather` function that always give back the same weather for all locations.

In [6]:
def calculator(a: str, b: str) -> float:
    return float(a) + float(b)

def get_weather(location: str) -> str:
    return f"Weather in {location}: Sunny, 72°F"

Each function is a `Tool` and as shown, can be as simple (or complex) as you can think off. More complexity, however, might result in your `TinyAgent` having difficulties understanding how it works. What's nice about the above examples is that the `calculator` is a bit misleading, perhaps `add` or something similar would have been prefered. As such, you also want to give your tool some form of description that can be used by your `TinyAgent`. 

## 4 - Tool Definition

Defining your tool can be done in various ways. Using the docstrings is typically a nice option. For our purpose, we are going to keep it simple and create a very short description for each of the tools. To do so, let's also build the `Tools` class that manages how tools are defined and used.

We start small and give it the following functions:

* `add_tool` -- Add a tool for the Agent to use
* `descriptions` -- The descriptions of each prompt
* `schemas` -- Used for native tool calling and for now is kept empty. In the second notebook `chapter05_native.ipynb`, we will cover how to let the LLM itself perform native tool calling without us having to define a schema beforehand.

In [7]:
from typing import Callable


class Tools:
    """Tool registry for the Agent."""

    def __init__(self):
        self.registry = {}

    def add_tool(self, name: str, func: Callable, description: str):
        """Register a tool that the Agent can use.

        Arguments:
            name: The name of the tool.
            func: The function implementing the tool.
            description: A description of the tool.
        """
        self.registry[name] = {"function": func, "description": description}

    @property
    def descriptions(self):
        """Get descriptions of all registered tools."""
        return "\n".join(
            f"`{tool}`: {self.registry[tool]['description']}" for tool in self.registry
        )

    @property
    def schemas(self):
        """Used only for native tool-calling."""
        return None


When can then add the Tools, including their descriptions as follows:

In [8]:
# Register tools
tools = Tools()
tools.add_tool("calculator", calculator, "Adds two numbers: calculator(a: str, b: str)")
tools.add_tool("get_weather", get_weather, "Gets weather: get_weather(location: str)")

Note that because these are simple functions, we do not need much for the LLM to understand. When the tools grow in complexity and number of parameters, however, a more extensive description is needed that details the type of parameters, what they do, and what kind of output is to be expected.

We can view the descriptions of our tools:

In [9]:
print(tools.descriptions)

`calculator`: Adds two numbers: calculator(a: str, b: str)
`get_weather`: Gets weather: get_weather(location: str)


As covered in the book, responsing with JSON is a nice trick for getting structured output for your `TinyAgent` to parse.

## 5 - Tool Selection


Having your `TinyAgent` select a tool to use requires telling your Agent that it has access to the tools and how to use them. As such, we have to add a `prompt` property that generates that prompt for us with additional instructions.


In [10]:
from typing import Callable


class Tools:
    """Tool registry for the Agent."""

    def __init__(self):
        self.registry = {}

    def add_tool(self, name: str, func: Callable, description: str):
        """Register a tool that the Agent can use.

        Arguments:
            name: The name of the tool.
            func: The function implementing the tool.
            description: A description of the tool.
        """
        self.registry[name] = {"function": func, "description": description}

    @property
    def descriptions(self):
        """Get descriptions of all registered tools."""
        return "\n".join(
            f"`{tool}`: {self.registry[tool]['description']}" for tool in self.registry
        )

    @property
    def prompt(self):
        return f"""
# Tools

If needed, you can only use the following tools to assist you in completing tasks:

{self.descriptions}

To use a tool, respond with JSON: {{"tool": "name", "kwargs": {{"param": "value"}}}}
"""


    @property
    def schemas(self):
        """Used only for native tool-calling."""
        return None

When we now create our `Tools`, we can use `prompt` to view the prompt that will be given to your `TinyAgent`:

In [11]:
tools = Tools()
tools.add_tool("calculator", calculator, "Adds two numbers: calculator(a: str, b: str)")
tools.add_tool("get_weather", get_weather, "Gets weather: get_weather(location: str)")
print(tools.prompt)


# Tools

If needed, you can only use the following tools to assist you in completing tasks:

`calculator`: Adds two numbers: calculator(a: str, b: str)
`get_weather`: Gets weather: get_weather(location: str)

To use a tool, respond with JSON: {"tool": "name", "kwargs": {"param": "value"}}



As covered in the book, responsing with JSON is a nice trick for getting structured output for your `TinyAgent` to parse.

## 6 - Tool Calling

In the previous step, the model will most likely do something like:

```python
response = '{"tool": "calculator", "kwargs": {"a": "value_1", "b": "value_2"} }'
```

We will have to parse this response into proper JSON and then call the tools ourselves. The `TinyAgent` therefore doesn't actually call the tool, it has to be converted first before it can actually do something.

Tool calling in `Tools` requires adding the following functions:

* `parse` -- Converts a string to JSON so that we have a structured tool call
* `execute` -- Run a tool call
* `observation` -- The output of a tool call
* `is_done` -- Checks whether there is a parsed tool call

We need to first check whether a string actually contains a tool call, we then parse the tool call, and finally run it.

In [12]:
import json
from typing import Callable

from illustrated_agents.chapters.ch2 import Response


class Tools:
    """Tool registry for the Agent."""

    def __init__(self):
        self.registry = {}

    def add_tool(self, name: str, func: Callable, description: str):
        """Register a tool that the Agent can use.

        Arguments:
            name: The name of the tool.
            func: The function implementing the tool.
            description: A description of the tool.
        """
        self.registry[name] = {"function": func, "description": description}

    @property
    def descriptions(self) -> str:
        """Get descriptions of all registered tools."""
        return "\n".join(
            f"`{tool}`: {self.registry[tool]['description']}" for tool in self.registry
        )

    @property
    def prompt(self) -> str:
        return f"""
# Tools

If needed, you can only use the following tools to assist you in completing tasks:

{self.descriptions}

To use a tool, respond with JSON: {{"tool": "name", "kwargs": {{"param": "value"}}}}
"""

    def parse(self, response: Response) -> Response:
        """Parse a JSON tool call from text."""
        text = response.content

        if '"tool":' in text or '"tool:"' in text:
            start, end = text.find("{"), text.rfind("}") + 1
            tool_call = json.loads(text[start:end])

            # Add the parsed tool call to the response
            return Response(
                content=response.content,
                reasoning=response.reasoning,
                tool_call=tool_call,
            )

        return response

    def execute(self, response: Response) -> any:
        """Run a registered tool.

        Arguments:
            tool_call: A parsed tool call dict with "tool" and "kwargs" keys.
        """
        tool_call = response.tool_call
        name, kwargs = tool_call["tool"], tool_call.get("kwargs", {})

        # Handle registered tools
        if name in self.registry:
            tool_func = self.registry[name]["function"]
            return tool_func(**kwargs)

        return f"Tool '{name}' not found."

    def observation(self, result):
        """Return the observation as a user."""
        return "user", f"OBSERVATION: {result}"

    def is_done(self, response: Response) -> bool:
        """The `TinyAgent` is done if there is no tool call."""
        return not response.tool_call

    @property
    def schemas(self):
        """Used only for native tool-calling."""
        return None

Let's go through each function step-by-step to check what it does. First, we start with parsing the response. This is a very simplified way of handling this and in practice you might want to use actual structured JSON instead. However, since you might be using an LLM that does not handle structured JSON properly, we do it through this (rather simplified) example.

In [13]:
from illustrated_agents.chapters.ch5 import parse_tool_annotated; parse_tool_annotated

If you expect a more complex JSON structure with nested dictionaries, this will not work and requires a more complex function that first cleans up the text (which is not needed when using structured JSON; see the Chapter in the book for more information). 

In [14]:
from illustrated_agents.chapters.ch5 import execute_tool_annotated; execute_tool_annotated

The `observation` function returns the output of the tool execution and will specify the role that should be occupied for that. Here, we are using `user` since there is no native tool calling (which can use the `tool` role):

In [15]:
from illustrated_agents.chapters.ch5 import observation_tool_annotated; observation_tool_annotated

The `is_done` functions will return the `Response.content` when it finds no tool calls:

In [16]:
from illustrated_agents.chapters.ch5 import done_tool_annotated; done_tool_annotated

## 7 - Tool Output Processing (updating `agent.py`)

Processing the output of a tool is handled by the LLM, so we can add that step to your `TinyAgent`. Note that the main differences are at:

* `system_prompt` -> A system prompt was added. 
* `._execute_action` -> A couple of lines of code to check if a tool was called. If True, then we parse and execute the tool. Since this is still a single-turn example, the output of the tool is returned to the user.

In [17]:
from illustrated_agents.chapters.ch2 import Trajectory
from illustrated_agents.chapters.ch4 import Memory


class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self, llm: LLM, memory: Memory, tools: Tools):
        self.llm = llm
        self.memory = memory
        self.tools = tools
        self.planner = None  # Chapter 6: Add Planning
        self.skills = None  # Chapter 6: Add Skills

        self.trajectory = Trajectory()

        # Build system prompt with all components
        system_prompt = "You are a helpful assistant.\n\n"
        system_prompt += self.tools.prompt
        self.memory.add("system", system_prompt)

    def run(self, task: str) -> str:
        """Run the agent on a task."""
        self.memory.add("user", task)
        self.trajectory.initialize(task)

        return self._step()

    def _step(self) -> str:
        """Perform a single step."""
        # THOUGHT: Generate response and add to memory
        response = self.llm.generate(self.memory.get_messages())
        self.memory.add("assistant", response.content)

        # Tool parsing
        response = self.tools.parse(response)

        # ANSWER: Stopping mechanism
        if self.tools.is_done(response):
            self.trajectory.add(response)
            return response.content

        return self._execute_action(response)

    def _execute_action(self, response: Response) -> None:
        """Execute a tool action."""

        # ACTION: execute tools
        result = self.tools.execute(response)

        # OBSERVATION: add tool results to memory and display
        role, observation = self.tools.observation(result)
        self.memory.add(role, observation)
        self.trajectory.add(response, observation)

        return observation

Let's go through these changes step-by-step, starting with the new `system_prompt` we created:

In [18]:
from illustrated_agents.chapters.ch5 import agent_init_annotated; agent_init_annotated

Next, we need to check whether there is actually a tool call in the response that was generated by the LLM:

In [19]:
from illustrated_agents.chapters.ch5 import agent_step_annotated; agent_step_annotated

Finally, the tool gets execute in `_execute_action`. This is a separate function that we created to include more than just tools in subsequent chapters, like executing SKILL.md and return intermediate answers. This function first parses the response so that only JSON is left (1) and then runs thes the tool (2) before adding the output to memory (3).

In [20]:
from illustrated_agents.chapters.ch5 import agent_execute_action_annotated; agent_execute_action_annotated

Here is an overview of the changes that we made to `agent.py`:

In [21]:
from illustrated_agents.chapters.ch5 import tinyagents_diff; tinyagents_diff

The `TinyAgent` can now be initialized with both the `Memory` and `Tools` modules:

In [22]:
# Tools
tools = Tools()
tools.add_tool("calculator", calculator, "Adds two numbers: calculator(a: str, b: str)")
tools.add_tool("get_weather", get_weather, "Gets weather: get_weather(location: str)")

# Memory
memory = Memory()

# Initialize Agent
agent = TinyAgent(llm=llm, memory=memory, tools=tools)

Let's check if the `TinyAgent` uses a tool when confronted with a question that may require one.

In [23]:
agent.run("What is 5.1281 plus 7.323?")

'OBSERVATION: 12.4511'

It sure did! Note that not all models will strictly follow instructions and will fail every so often. Either way, the prerendered output shows that a tool was correctly used!

Let's explore the traces of the interaction in case something went wrong:

In [24]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

Let's also explore the Agent's memory:

In [25]:
agent.memory.get_messages()

[{'role': 'system',
  'content': 'You are a helpful assistant.\n\n\n# Tools\n\nIf needed, you can only use the following tools to assist you in completing tasks:\n\n`calculator`: Adds two numbers: calculator(a: str, b: str)\n`get_weather`: Gets weather: get_weather(location: str)\n\nTo use a tool, respond with JSON: {"tool": "name", "kwargs": {"param": "value"}}\n'},
 {'role': 'user', 'content': 'What is 5.1281 plus 7.323?'},
 {'role': 'assistant',
  'content': '{"tool": "calculator", "kwargs": {"a": "5.1281", "b": "7.323"}}\n'},
 {'role': 'user', 'content': 'OBSERVATION: 12.4511'}]

We can check if your `TinyAgent` will answer questions that do not require tool usage:

In [26]:
agent.run("Hi! Tell me something about flamingos in two sentences.")

'Flamingos are vibrant pink birds that get their color from the pigments in the algae and crustaceans they eat. These fascinating birds are often found in large flocks in tropical and subtropical regions around the world.\n'

It does! 😁 LLMs these days (Jan. 2026) are great in deciding themselves which tool to use and when. However, that does not mean it is not fallible. Describing hundreds of tools will likely fill up the context window too much and make it difficult for the LLM to decide if to use a tool and which one. The updated trajectory shows this:

In [27]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered the various stages of `Tools`, like defining, parsing, and running tools. The result is two new files: `tools.py` for the main class and `toolbox.py` where we keep a reference of the tools that were created.

In [28]:
from illustrated_agents.chapters.ch5 import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py      ← Updated (Added `Tools`!)                                                                    │
│ ├── llm.py                                                                                                      │
│ ├── memory.py                                                                                                   │
│ ├── toolbox.py    ← New (This file tracks all tools that were created for easy reference.)                      │
│ ├── tools.py      ← New (Created a tool registry, parsing, and execution class.)                                │
│ └── trajectory.py                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# What's Next

**Native Tool Calling**

Now that you learned how to perform explicit tool calling through prompt engineering, you are ready for the next step! Please visit `chapter05_native_tool_calling.ipynb` to explore how to perform native tool calling with Gemma 4 E4B. The prompt engineering you did has evolved to become part of the training data. Instead of having to create specific prompts for specific tool calling capabilities, the model was trained with specific tokens to that itself. This is much more stable and tends to improve the stability of the model's capability to call tools.

**Model Context Protocol (MCP)**

To learn how to use Model Context Protocol (MCP) for standardized tool calling capabilities, please explore `chapter05_mcp.ipynb`. It will be an in-depth guide on how to setup your own server and create your own set of tools.